# TUẦN 4: TINH CHỈNH (FINE-TUNING) VINAI PHOBERT V2 TRÊN GPU
## (ONE-CLICK HIGH-PERFORMANCE TRAINING NOTEBOOK FOR COLAB & LOCAL)

> **Đồ án:** Phân tích Cảm xúc Đánh giá Khách hàng Shopee (Vietnamese E-commerce Sentiment Analysis)  
> **Học phần:** Artificial Intelligence Capstone 2025 • Viện VNUK, Đại học Đà Nẵng  
> **Phụ trách:** Từ Văn Huy Hoàng (Senior ML / Deep Learning Engineer)  
> **Chỉ tiêu kỹ thuật (KPIs):**  
> - **RQ2 (Model Benchmark):** Phá vỡ mốc trần Baseline, đạt **Macro F1 >= 75.0%** trên tập kiểm thử độc lập.  
> - **RQ3 (Class Imbalance):** Áp dụng Weighted Cross-Entropy Loss (w_c = N / (K * N_c)) để giải cứu lớp thiểu số Trung tính (NEU).  
> - **Cơ chế One-Click:** Tự động phát hiện môi trường, tự clone/tải dữ liệu, tự huấn luyện và tự kích hoạt tải file `best_phobert.pt` về máy tính khi hoàn tất.

## 1. THIẾT LẬP MÔI TRƯỜNG & TỰ ĐỘNG CÀI ĐẶT THƯ VIỆN

In [ ]:
# Tự động phát hiện môi trường Google Colab
import os
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print('🚀 Phát hiện đang chạy trên Google Colab!')
    !pip install -q transformers pyvi scikit-learn seaborn matplotlib tqdm
    if not os.path.exists('Project_Intro_AI'):
        !git clone -b develop https://github.com/nguyenduythaibao1611-eng/Project_Intro_AI.git
        os.chdir('Project_Intro_AI')
    ROOT_DIR = Path.cwd()
else:
    ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import torch
import random
import numpy as np

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️ Thiết bị tính toán: {device}')
if torch.cuda.is_available():
    print(f'🔥 GPU Tên: {torch.cuda.get_device_name(0)}')
    print(f'💾 VRAM Khả dụng: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('⚠️ Lưu ý: Đang chạy trên CPU. Khuyến nghị bật GPU T4 miễn phí trên Google Colab để train trong 15 phút!')


## 2. KIỂM TRA & NẠP DỮ LIỆU ĐÃ CHUẨN HÓA

In [ ]:
import pandas as pd

path_data = ROOT_DIR / 'data' / 'processed'
train_csv = path_data / 'train.csv'
val_csv = path_data / 'val.csv'
test_csv = path_data / 'test.csv'

# Tự động sinh dữ liệu phân tầng nếu chưa có
if not (train_csv.exists() and val_csv.exists() and test_csv.exists()):
    print('⚙️ Đang thực hiện chia tách tập dữ liệu phân tầng (Stratified Split)...\n')
    from src.preprocess import split_and_save_stratified
    raw_clean = path_data / 'shopee_reviews_cleaned.csv'
    split_and_save_stratified(str(raw_clean), str(path_data), random_state=RANDOM_SEED)
    print('✅ Đã tạo xong train.csv, val.csv, test.csv!')

df_train = pd.read_csv(train_csv)
df_val = pd.read_csv(val_csv)
df_test = pd.read_csv(test_csv)

print(f'📊 Số mẫu tập Train: {len(df_train):,} mẫu')
print(f'📊 Số mẫu tập Val:   {len(df_val):,} mẫu')
print(f'📊 Số mẫu tập Test:  {len(df_test):,} mẫu')


## 3. THIẾT KẾ PYTORCH DATASET & DATALOADER CHO TRANSFORMER

In [ ]:
from transformers import AutoTokenizer
from torch.utils.data import DataLoader
from src.dataset import MAX_SEQUENCE_LENGTH
from src.train import PhoBERTReviewDataset

MODEL_NAME = 'vinai/phobert-base-v2'
print(f'📥 Đang nạp Tokenizer: {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = PhoBERTReviewDataset(
    texts=df_train['processed_content'].tolist(),
    labels=df_train['label_id'].tolist(),
    tokenizer=tokenizer,
    max_length=MAX_SEQUENCE_LENGTH,
)
val_dataset = PhoBERTReviewDataset(
    texts=df_val['processed_content'].tolist(),
    labels=df_val['label_id'].tolist(),
    tokenizer=tokenizer,
    max_length=MAX_SEQUENCE_LENGTH,
)
test_dataset = PhoBERTReviewDataset(
    texts=df_test['processed_content'].tolist(),
    labels=df_test['label_id'].tolist(),
    tokenizer=tokenizer,
    max_length=MAX_SEQUENCE_LENGTH,
)

BATCH_SIZE = 32 if torch.cuda.is_available() else 8
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f'✅ Đã tạo DataLoader chuẩn. Số batch train: {len(train_loader)} (Batch Size = {BATCH_SIZE})')


## 4. KHỞI TẠO PHOBERT V2, WEIGHTED LOSS (RQ3) & ADAMW PHÂN TẦNG
- **Backbone:** learning rate = `2e-5`
- **Classification Head:** learning rate = `1e-4` (Tránh Mode Collapse sang NEU)
- **Scheduler:** Linear Warmup trong 10% số bước đầu tiên.

In [ ]:
from src.models import (
    PhoBERTClassifier,
    compute_class_weights,
    get_weighted_criterion,
    get_phobert_optimizer_and_scheduler,
)

model_phobert = PhoBERTClassifier(
    model_name=MODEL_NAME,
    num_classes=3,
    dropout_rate=0.2,
).to(device)

# Tính toán trọng số phân lớp giải cứu lớp NEU (RQ3)
class_weights = compute_class_weights(df_train['label_id'].values, num_classes=3, device=device)
criterion = get_weighted_criterion(class_weights, device=device)
print(f'⚖️ Class weights (RQ3): {class_weights.cpu().numpy().tolist()}')

EPOCHS = 4
total_steps = len(train_loader) * EPOCHS
optimizer, scheduler = get_phobert_optimizer_and_scheduler(
    model=model_phobert,
    total_steps=total_steps,
    lr_backbone=2e-5,
    lr_head=1e-4,  # Chuẩn hóa 1e-4 triệt tiêu hiện tượng giật gradient
    warmup_ratio=0.1,
    weight_decay=0.01,
)
print(f'✅ Khởi tạo thành công AdamW phân tầng & Linear Warmup ({int(total_steps*0.1)} warmup steps)')


## 5. TIẾN HÀNH HUẤN LUYỆN 4 EPOCHS & LƯU CHECKPOINT TỐI ƯU

In [ ]:
import time
from src.models import train_one_epoch, evaluate_model, save_checkpoint
from src.evaluate import compute_classification_metrics

best_checkpoint_path = ROOT_DIR / 'models' / 'best_phobert.pt'
best_checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
best_val_f1 = -1.0

print(f'🚀 BẮT ĐẦU FINE-TUNING PHOBERT V2 TRONG {EPOCHS} EPOCHS TRÊN {device}...\n' + '='*75)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss, train_acc = train_one_epoch(
        model=model_phobert,
        dataloader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        clip_grad=1.0,
        scheduler=scheduler,
    )
    val_loss, val_acc, y_val_t, y_val_p = evaluate_model(
        model=model_phobert,
        dataloader=val_loader,
        criterion=criterion,
        device=device,
    )
    val_metrics = compute_classification_metrics(y_val_t, y_val_p)
    val_macro_f1 = float(val_metrics['macro_f1'])
    elapsed = time.time() - t0
    
    print(f'Epoch [{epoch:02d}/{EPOCHS:02d}] ({elapsed:.1f}s) | '
          f'Train Loss: {train_loss:.4f} Acc: {train_acc*100:.2f}% | '
          f'Val Loss: {val_loss:.4f} Acc: {val_acc*100:.2f}% | '
          f'Val Macro F1: {val_macro_f1*100:.2f}%')
    
    if val_macro_f1 > best_val_f1:
        best_val_f1 = val_macro_f1
        save_checkpoint(
            model=model_phobert,
            optimizer=optimizer,
            epoch=epoch,
            metrics={'val_macro_f1': val_macro_f1, 'val_acc': val_acc},
            save_path=best_checkpoint_path,
            extra_state={
                'model_type': 'phobert',
                'model_name': MODEL_NAME,
                'num_classes': 3,
                'dropout_rate': 0.2,
                'class_weights': class_weights.cpu().tolist(),
            }
        )
        print(f'   --> 🌟 ĐÃ LƯU CHECKPOINT TỐI ƯU MỚI: Val Macro F1 = {val_macro_f1*100:.2f}%\n')


## 6. THẨM ĐỊNH TRÊN TẬP TEST ĐỘC LẬP (ĐÁNH GIÁ RQ2 SOTA BENCHMARK)

In [ ]:
from src.models import load_checkpoint

print('🔎 Đang nạp lại checkpoint tối ưu để đánh giá trên tập kiểm thử độc lập...')
model_phobert, ckpt_info = load_checkpoint(best_checkpoint_path, model=model_phobert, device=device)

test_loss, test_acc, y_test_t, y_test_p = evaluate_model(
    model=model_phobert,
    dataloader=test_loader,
    criterion=criterion,
    device=device,
)
test_metrics = compute_classification_metrics(y_test_t, y_test_p)

print('\n' + '='*60)
print(f'🏆 BÁO CÁO HIỆU NĂNG PHOBERT V2 TRÊN TẬP TEST ĐỘC LẬP ({len(df_test):,} MẪU):')
print(f' - Độ chính xác toàn thể (Accuracy): {test_metrics["accuracy"]*100:.2f}%')
print(f' - Macro Precision:                   {test_metrics["macro_precision"]*100:.2f}%')
print(f' - Macro Recall:                      {test_metrics["macro_recall"]*100:.2f}%')
print(f' - Macro F1-score (Chỉ tiêu >= 75%): {test_metrics["macro_f1"]*100:.2f}%')
print(f' - Chi tiết từng lớp:')
for cls_name, f1 in test_metrics["per_class"].items():
    recall = test_metrics["per_class_recall"].get(cls_name, 0.0)
    print(f'   + Lớp {cls_name}: F1 = {f1*100:.2f}% | Recall = {recall*100:.2f}%')
print('='*60)


## 7. TỰ ĐỘNG TẢI FILE CHECKPOINT VỀ MÁY TÍNH (CHO GOOGLE COLAB)

In [ ]:
if IN_COLAB:
    from google.colab import files
    print('📥 Đang kích hoạt tải tự động best_phobert.pt về máy...')
    files.download(str(best_checkpoint_path))
    print('🎉 XONG! Hãy chuyển file vừa tải vào thư mục models/ của dự án.')
else:
    print(f'✅ Tệp checkpoint tối ưu đã được lưu sẵn tại: {best_checkpoint_path}')
